# Senologie-Kohortenanalyse (Aidbox)

Dieses Notebook demonstriert die Auswertung der synthetischen Senologie-Kohorte (12 Patientinnen, ~290 Ressourcen) über die sechs **SQL-on-FHIR v2 `ViewDefinition`**-Artefakte des IG.

## Backend

**Aidbox** führt die ViewDefinitions nativ über `ViewDefinition/<name>/$run` aus — kein externer Runner nötig.

> Für die Pathling-Variante (Docker/Python/Custom-Runner) siehe `senologie-analyse.ipynb`.

### Voraussetzungen

```bash
docker compose up -d          # Aidbox + PostgreSQL starten
./scripts/import-to-aidbox.sh # Profile, Beispiele, ViewDefinitions laden
pip install pandas requests matplotlib seaborn
```

## 0. Setup

In [ ]:
import json
from datetime import date
from typing import Dict, List

import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='deep')
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

# --- Aidbox-Verbindung -------------------------------------------------------
AIDBOX_URL = 'http://localhost:8888'
AUTH = HTTPBasicAuth('root', 'secret')

# Health-Check
r = requests.get(f'{AIDBOX_URL}/health', timeout=10)
r.raise_for_status()
print(f'Aidbox erreichbar: {AIDBOX_URL} (Status: {r.json()["status"]})')

### 0a. ViewDefinitions aus Aidbox laden

Die Views sind bereits über `import-to-aidbox.sh` als `ViewDefinition`-Ressourcen in Aidbox gespeichert.

In [ ]:
VIEW_NAMES = [
    'DiagnoseKohorte',
    'OperationenKohorte',
    'PathologieKohorte',
    'PatientKohorte',
    'SystemtherapieKohorte',
    'TumorboardKohorte',
]

# Prüfe, dass alle Views in Aidbox vorhanden sind
for name in VIEW_NAMES:
    r = requests.get(f'{AIDBOX_URL}/ViewDefinition/{name}', auth=AUTH, timeout=10)
    symbol = 'ok' if r.status_code == 200 else f'FEHLT (HTTP {r.status_code})'
    print(f'  {name:28s}  {symbol}')

## 1. Views materialisieren

Aidbox führt `ViewDefinition/$run` serverseitig aus und liefert flache JSON-Arrays zurück.

In [ ]:
def run_view_aidbox(name: str) -> pd.DataFrame:
    """Execute a named ViewDefinition via Aidbox $run and return as DataFrame."""
    r = requests.post(
        f'{AIDBOX_URL}/fhir/ViewDefinition/{name}/$run',
        auth=AUTH,
        headers={
            'Content-Type': 'application/fhir+json',
            'Accept': 'application/json',
        },
        json={
            'resourceType': 'Parameters',
            'parameter': [{'name': '_format', 'valueCode': 'json'}],
        },
        timeout=60,
    )
    r.raise_for_status()
    rows = r.json()
    return pd.DataFrame(rows) if isinstance(rows, list) else pd.DataFrame()

df: Dict[str, pd.DataFrame] = {}
for name in VIEW_NAMES:
    df[name] = run_view_aidbox(name)
    print(f"{name:28s}  rows={len(df[name]):4d}  cols={len(df[name].columns)}")

# patientId normalisieren (Patient/-Prefix entfernen falls vorhanden)
for key in ('DiagnoseKohorte', 'OperationenKohorte', 'PathologieKohorte', 'SystemtherapieKohorte', 'TumorboardKohorte'):
    if key in df and 'patientId' in df[key].columns:
        df[key]['patientId'] = df[key]['patientId'].astype(str).str.replace(r'^Patient/', '', regex=True)

In [ ]:
df['PatientKohorte'].head(20)

In [ ]:
df['DiagnoseKohorte'].head(20)

In [ ]:
df['OperationenKohorte'].head(20)

In [ ]:
df['PathologieKohorte'].head(20)

In [ ]:
df['SystemtherapieKohorte'].head(20)

In [ ]:
df['TumorboardKohorte'].head(20)

## 5. Ableitung: Alter, Subtyp, BET-Art

Ein paar heuristische Ableitungen, die auch ohne strukturierte IHC-Observations aus dem Freitext rekonstruiert werden:

In [ ]:
# Alter aus birthDate berechnen
pat = df['PatientKohorte'].copy()
pat['birthDate'] = pd.to_datetime(pat['birthDate'], errors='coerce')
today_dt = pd.Timestamp(date.today())
pat['age'] = ((today_dt - pat['birthDate']).dt.days / 365.25).round(1)
df['PatientKohorte'] = pat

# Subtyp aus Pathologie + Diagnose ableiten
patho = df['PathologieKohorte'].copy()
patho['datum'] = pd.to_datetime(patho['datum'], errors='coerce')

diag = df['DiagnoseKohorte'].copy()
# DCIS-Markierung
diag['isDCIS'] = diag['snomed'].astype(str).isin(['109889007', '254838004']) | diag['icd10'].astype(str).str.startswith('D05')

# Subtyp pro Patient: aus dem _letzten_ Pathologiebefund pro Patient
patho_sorted = patho.sort_values(['patientId', 'datum']).groupby('patientId').tail(1)

def subtype_for(row) -> str:
    er, pr, her2 = row.get('er'), row.get('pr'), row.get('her2')
    if pd.isna(er) and pd.isna(pr) and pd.isna(her2):
        return 'unbekannt'
    if her2 == 'positive':
        return 'HER2+'
    if (er == 'negative' and pr == 'negative' and her2 == 'negative'):
        return 'TNBC'
    if (er == 'positive' or pr == 'positive') and (her2 in ('negative', None, float('nan'))):
        return 'HR+/HER2-'
    return 'unklar'

patho_sorted['subtyp'] = patho_sorted.apply(subtype_for, axis=1)

# DCIS überschreibt (reine DCIS-Fälle gehen vor)
dcis_patients = set(diag[diag['isDCIS']]['patientId'].unique()) - set(
    diag[(~diag['isDCIS']) & (diag['snomed'] == '254837009')]['patientId'].unique()
)
pat_subtype = patho_sorted.set_index('patientId')['subtyp']
for pid in dcis_patients:
    pat_subtype[pid] = 'DCIS'

# Patienten ohne Patho aber mit DCIS-Diagnose
for pid in dcis_patients:
    if pid not in pat_subtype.index:
        pat_subtype[pid] = 'DCIS'

df['PatientKohorte']['subtyp'] = df['PatientKohorte']['id'].map(pat_subtype).fillna('unbekannt')
df['PatientKohorte'][['id', 'name', 'age', 'subtyp']].head(15)

## Analyse 1 — Fallzahl pro Subtyp

Verteilung der Patientinnen auf die vier Haupt-Subtypen (HR+/HER2-, HER2+, TNBC, DCIS).

In [ ]:
subtype_counts = df['PatientKohorte']['subtyp'].value_counts()
print(subtype_counts)

fig, ax = plt.subplots(figsize=(7, 4))
subtype_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato', 'goldenrod', 'mediumseagreen', 'slategray', 'lightgray'])
ax.set_title('Fallzahl pro Subtyp')
ax.set_xlabel('Subtyp')
ax.set_ylabel('Patientinnen')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 2 — BET-Rate

Anteil brusterhaltender Operationen an allen Brust-OPs (BET vs. Mastektomie). DKG-Qualitätsindikator.

In [ ]:
op = df['OperationenKohorte'].copy()
brust_ops = op[op['art'].isin(['BET', 'Mastektomie'])]
print(f'Brust-Operationen gesamt: {len(brust_ops)}')

bet_count = int((brust_ops['art'] == 'BET').sum())
mast_count = int((brust_ops['art'] == 'Mastektomie').sum())
bet_rate = bet_count / max(1, bet_count + mast_count) * 100
print(f'BET: {bet_count}, Mastektomie: {mast_count}')
print(f'BET-Rate: {bet_rate:.1f}%')

fig, ax = plt.subplots(figsize=(5, 4))
brust_ops['art'].value_counts().plot(kind='bar', ax=ax, color=['seagreen', 'coral'])
ax.set_title(f'BET-Rate: {bet_rate:.1f}%')
ax.set_xlabel('OP-Art')
ax.set_ylabel('Anzahl')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 3 — R0-Rate

Anteil R0-Resektionen an allen dokumentierten Brust-OPs. Qualitätsindikator QI-11 (Ziel: möglichst hoch).

In [ ]:
op_with_r = op[op['outcome'].notna()]
print(f'OPs mit R-Status dokumentiert: {len(op_with_r)} von {len(op)} gesamt')

r_counts = op_with_r['outcome'].value_counts()
print(r_counts)

r0_rate = (op_with_r['outcome'] == 'R0').sum() / max(1, len(op_with_r)) * 100
print(f'R0-Rate: {r0_rate:.1f}%')

fig, ax = plt.subplots(figsize=(5, 4))
r_counts.plot(kind='bar', ax=ax, color=['mediumseagreen', 'orange', 'firebrick'])
ax.set_title(f'R-Status Verteilung (R0-Rate: {r0_rate:.1f}%)')
ax.set_xlabel('R-Status')
ax.set_ylabel('Anzahl')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 4 — Altersverteilung

Boxplot der Patientinnenalter, optional stratifiziert nach Subtyp.

In [ ]:
pat_age = df['PatientKohorte'].dropna(subset=['age']).copy()
print(pat_age[['age']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(pat_age['age'], vert=True, patch_artist=True,
                 boxprops=dict(facecolor='lightsteelblue'))
axes[0].set_title('Altersverteilung (alle)')
axes[0].set_ylabel('Alter in Jahren')
axes[0].set_xticklabels(['Gesamt'])

if HAS_SNS:
    sns.boxplot(data=pat_age, x='subtyp', y='age', ax=axes[1])
else:
    groups = pat_age.groupby('subtyp')['age'].apply(list)
    axes[1].boxplot(groups.values, labels=groups.index)
axes[1].set_title('Alter nach Subtyp')
axes[1].set_xlabel('Subtyp')
axes[1].set_ylabel('Alter in Jahren')

plt.tight_layout()
plt.show()

## Analyse 5 — Therapie-Mix pro Subtyp

Welche Systemtherapie-Art (CH = Chemo, HO = Hormon, IM = Immun, ZT = Zielgerichtet) pro Subtyp dokumentiert ist.

In [ ]:
sys = df['SystemtherapieKohorte'].copy()
sys = sys.merge(df['PatientKohorte'][['id', 'subtyp']].rename(columns={'id': 'patientId'}),
                on='patientId', how='left')

therapy_mix = pd.crosstab(sys['subtyp'], sys['art']).fillna(0).astype(int)
print(therapy_mix)

fig, ax = plt.subplots(figsize=(8, 4))
therapy_mix.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Therapie-Mix pro Subtyp')
ax.set_xlabel('Subtyp')
ax.set_ylabel('Anzahl Therapien')
ax.legend(title='Therapie', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 6 — Time-to-Treatment

Verteilung der Zeitspanne von Diagnose bis zur Erst-Operation (in Tagen). Relevanter Indikator der Versorgungsqualität.

In [ ]:
diag2 = df['DiagnoseKohorte'].copy()
diag2['diagnosedatum'] = pd.to_datetime(diag2['diagnosedatum'], errors='coerce')
diag2['onsetDateTime'] = pd.to_datetime(diag2['onsetDateTime'], errors='coerce')
diag2['recordedDate'] = pd.to_datetime(diag2['recordedDate'], errors='coerce')
diag2['diag_datum'] = diag2['diagnosedatum'].fillna(diag2['onsetDateTime']).fillna(diag2['recordedDate'])

first_diag = diag2.sort_values(['patientId', 'diag_datum']).groupby('patientId')['diag_datum'].first()

op2 = df['OperationenKohorte'].copy()
op2['datum'] = pd.to_datetime(op2['datum'].fillna(op2['beginn']), errors='coerce')
first_op = op2[op2['art'].isin(['BET', 'Mastektomie'])].sort_values(['patientId', 'datum']).groupby('patientId')['datum'].first()

tt = pd.concat({'diagnose': first_diag, 'op': first_op}, axis=1).dropna()
tt['tage'] = (tt['op'] - tt['diagnose']).dt.days
tt = tt[tt['tage'] >= 0]
print(tt[['tage']].describe().round(1))
print(f'\nMedian Time-to-Treatment: {tt["tage"].median():.0f} Tage')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tt['tage'], bins=12, color='steelblue', edgecolor='white')
ax.axvline(tt['tage'].median(), color='red', linestyle='--', label=f'Median {tt["tage"].median():.0f}d')
ax.axvline(28, color='gray', linestyle=':', label='Richtwert 28 Tage')
ax.set_title('Time-to-Treatment: Diagnose → Erst-OP')
ax.set_xlabel('Tage')
ax.set_ylabel('Anzahl Patientinnen')
ax.legend()
plt.tight_layout()
plt.show()

## Analyse 7 — Crosstab Subtyp × OP-Art

Welcher Subtyp erhält welche OP? Erwartet: DCIS → vermehrt BET, HER2+/TNBC → häufiger Mastektomie in höheren Stadien.

In [ ]:
op3 = df['OperationenKohorte'].copy()
op3 = op3.merge(df['PatientKohorte'][['id', 'subtyp']].rename(columns={'id': 'patientId'}),
                on='patientId', how='left')
op3 = op3[op3['art'].isin(['BET', 'Mastektomie', 'SNB', 'ALND'])]

ct = pd.crosstab(op3['subtyp'], op3['art']).fillna(0).astype(int)
print(ct)

fig, ax = plt.subplots(figsize=(8, 4))
if HAS_SNS:
    sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', ax=ax, cbar=False)
else:
    im = ax.imshow(ct.values, cmap='YlGnBu', aspect='auto')
    for (i, j), v in pd.DataFrame(ct).stack().items():
        ax.text(list(ct.columns).index(j), list(ct.index).index(i), str(v), ha='center', va='center')
    ax.set_xticks(range(len(ct.columns)), ct.columns)
    ax.set_yticks(range(len(ct.index)), ct.index)
ax.set_title('OP-Art × Subtyp')
plt.tight_layout()
plt.show()

## Zusammenfassung

Dieses Notebook nutzt **Aidbox** als SQL-on-FHIR-Engine. Die ViewDefinitions werden serverseitig über `ViewDefinition/$run` ausgeführt — kein lokaler FHIRPath-Runner nötig.

Die Analyse zeigt exemplarisch, wie die SQL-on-FHIR-Views als Grundlage für:

- Qualitätsindikatoren (BET-Rate, R0-Rate, Time-to-Treatment)
- Subtypen-Stratifizierung (HR+/HER2-, HER2+, TNBC, DCIS)
- Therapie-Mix-Analysen
- Versorgungspfad-Analysen

verwendet werden können.

### Aidbox-spezifische Features

Zusätzlich zu den Views bietet Aidbox:
- **`/$sql`** — direktes SQL auf die FHIR-Datenbank
- **Aidbox Notebooks** — interaktive Abfragen in der Web-UI (`/ui/notebook`)
- **AidboxQuery** — vordefinierte parametrisierte Abfragen (z.B. `senologie-diagnose`)

### Vergleich mit Pathling-Variante

Für die Pathling-basierte Analyse (Docker/Python/Custom-Runner) siehe `senologie-analyse.ipynb`.